# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description from the metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id` values.

`mlcroissant` exposes schema via `dataset.metadata`. We'll enumerate available record sets (tables), then explore their fields and columns, referencing everything by their `@id`.

In [ ]:
# List all available record sets by their @id

record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")

for i, rs in enumerate(record_sets):
    print(f"{i+1}. @id: {rs['@id']}")
    print(f"   name: {rs.get('name')}")
    if 'description' in rs:
        print(f"   description: {rs['description']}")
    print("")
    
    fields = rs.get('fields', [])
    print(f"   Fields (by @id):")
    for f in fields:
        print(f"     - @id: {f['@id']}; name: {f.get('name')}; dataType: {f.get('dataType')}")
    print("")


## 3. Data Extraction
Load data from the main record set(s). Use `@id` values to select and reference them.

This example extracts all records from all available record sets and loads them as DataFrames for exploration.

In [ ]:
# Extract data from each record set
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    print(f"  Record count: {len(records)}")
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display first DataFrame info
if record_set_ids:
    default_rs_id = record_set_ids[0]
    print(f"Columns in record set '{default_rs_id}':")
    print(dataframes[default_rs_id].columns.tolist())
    display(dataframes[default_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps, such as filtering records based on a numeric field, normalizing, and grouping data. 

*Choose one suitable numeric field and one grouping field (`@id`) as examples, using the first record set for demonstration.*

In [ ]:
# Identify a numeric field and a grouping field by their @id.
rs = dataset.metadata.record_sets[0]  # Use the first record set.
numeric_field_id = None
group_field_id = None

for f in rs.get('fields', []):
    dt = str(f.get('dataType', ''))
    if numeric_field_id is None and (dt.lower() in {'float', 'number', 'integer'} or 'number' in dt.lower() or 'float' in dt.lower()):
        numeric_field_id = f['@id']  # Select the first numeric field
    if group_field_id is None and dt.lower() == 'text':
        group_field_id = f['@id']  # Select the first text field as group
    if numeric_field_id and group_field_id:
        break

print(f"Chosen numeric field (by @id): {numeric_field_id}")
print(f"Chosen group field (by @id): {group_field_id}")

df = dataframes[rs['@id']]

# Some CSVs may store numbers as strings; coerce to numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter: select records where the numeric field > threshold
threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records (first 5 rows):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field (if present)
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to the grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group field
if group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² dataset containing human protein mass spectrometry results. We programmatically discovered available record sets, fields, and extracted example statistics and visualizations using field `@id` references to ensure traceability. For a more detailed analysis, see the dataset documentation and expand EDA as appropriate.